# Multilingual Government Scheme Assistant using RAG

## Objective

This project builds a multilingual Retrieval-Augmented Generation (RAG)
system that allows users to ask questions about Indian government schemes
in English, Marathi, and Hindi.

The system retrieves relevant information from official government scheme
documents and generates answers with document name and page-level citations.

- English, Marathi and Hindi question support
- PDF-based knowledge base
- Multilingual embeddings
- FAISS vector database
- Query expansion
- Relevant document retrieval
- Reranking
- Source and page-level citations
- Context-grounded answers
- Fallback for unavailable information

In [40]:
#!pip install -U sentence-transformers langchain-huggingface

Defaulting to user installation because normal site-packages is not writeable


In [1]:
import os
import langchain
import warnings
warnings.filterwarnings('ignore')
os.environ["HUGGINGFACEHUB_API_TOKEN"] = "" #  put your id

In [2]:
from langchain_community.document_loaders import PyPDFLoader  # it is used to read the pdf

In [3]:
#  step 1  connect to LLM in this case--> Connect to DeepSeek-V3
from langchain_huggingface import HuggingFaceEndpoint ,  ChatHuggingFace
llm = HuggingFaceEndpoint(
    repo_id="deepseek-ai/DeepSeek-V3",    #  repo_id is used to tell which model to be used  LLM  model
    max_new_tokens= 512,                   # max number of words at a time
    temperature= .2,                      #  more  temp more creativity, less temp more reliablity
)
chat_model = ChatHuggingFace(llm=llm)

print("✅ DeepSeek-V3 LLM connected successfully!")

✅ DeepSeek-V3 LLM connected successfully!


In [4]:
import os
from langchain_community.document_loaders import PyPDFDirectoryLoader

# Government scheme PDF folder
pdf_folder = r"C:\Users\Admin\OneDrive\Documents\End_to_End_Project\Documents"

# Load all PDF files from the folder
loader = PyPDFDirectoryLoader(pdf_folder)
docs = loader.load()
print("Total pages loaded:", len(docs))

# List all PDF files in the folder
pdf_files = [
    file for file in os.listdir(pdf_folder)
    if file.lower().endswith(".pdf")
]

print("\nPDF files in the folder:")

for i, file in enumerate(pdf_files, start=1):
    print(f"{i}. {file}")

print(f"\nTotal PDF files: {len(pdf_files)}")
print(f"Total pages/documents loaded: {len(docs)}")

Ignoring wrong pointing object 24 0 (offset 0)
Ignoring wrong pointing object 34 0 (offset 0)
Ignoring wrong pointing object 49 0 (offset 0)
Ignoring wrong pointing object 87 0 (offset 0)
Ignoring wrong pointing object 114 0 (offset 0)
Ignoring wrong pointing object 170 0 (offset 0)
Ignoring wrong pointing object 254 0 (offset 0)
Ignoring wrong pointing object 272 0 (offset 0)


Total pages loaded: 648

PDF files in the folder:
1. NAIS_Scheme.pdf
2. New_Agriculture_Schemes_Maharashtra.pdf
3. PMAY_Urban_Guidelines.pdf
4. PMFBY_Revamped_OGs.pdf
5. PMJDY_Guidelines.pdf
6. PMJJBY_Rules.pdf
7. PMKVY_4_0.pdf
8. PM_KISAN_Guidelines.pdf
9. PM_KISAN_Operational_Guidelines.pdf
10. Revamped Operational Guidelines_17th August 2020.pdf
11. SDRFNDRFGuideline_20082024.pdf
12. WBCIS_Guidelines.pdf
13. WBCIS_Operational_Guidelines.pdf

Total PDF files: 13
Total pages/documents loaded: 648


In [5]:
from collections import Counter
page_counts = Counter(doc.metadata['source'] for doc in docs)
for src, count in page_counts.items():
    print(f"{src} -> {count} pages")

C:\Users\Admin\OneDrive\Documents\End_to_End_Project\Documents\NAIS_Scheme.pdf -> 34 pages
C:\Users\Admin\OneDrive\Documents\End_to_End_Project\Documents\New_Agriculture_Schemes_Maharashtra.pdf -> 38 pages
C:\Users\Admin\OneDrive\Documents\End_to_End_Project\Documents\PMAY_Urban_Guidelines.pdf -> 112 pages
C:\Users\Admin\OneDrive\Documents\End_to_End_Project\Documents\PMFBY_Revamped_OGs.pdf -> 144 pages
C:\Users\Admin\OneDrive\Documents\End_to_End_Project\Documents\PMJDY_Guidelines.pdf -> 1 pages
C:\Users\Admin\OneDrive\Documents\End_to_End_Project\Documents\PMJJBY_Rules.pdf -> 4 pages
C:\Users\Admin\OneDrive\Documents\End_to_End_Project\Documents\PMKVY_4_0.pdf -> 68 pages
C:\Users\Admin\OneDrive\Documents\End_to_End_Project\Documents\PM_KISAN_Guidelines.pdf -> 12 pages
C:\Users\Admin\OneDrive\Documents\End_to_End_Project\Documents\PM_KISAN_Operational_Guidelines.pdf -> 12 pages
C:\Users\Admin\OneDrive\Documents\End_to_End_Project\Documents\Revamped Operational Guidelines_17th August 2

In [6]:
import pytesseract

pytesseract.pytesseract.tesseract_cmd = (
    r"C:\Program Files\Tesseract-OCR\tesseract.exe"
)

print(pytesseract.get_tesseract_version())

5.5.3.20260724


In [7]:
# OCR (Optical Character Recognition)

import pymupdf
import pytesseract
from PIL import Image
import io

pdf_path =  r"C:\Users\Admin\OneDrive\Documents\End_to_End_Project\Documents\PMJDY_Guidelines.pdf"

pdf = pymupdf.open(pdf_path)

ocr_text = ""

for page_num, page in enumerate(pdf):

    pix = page.get_pixmap(matrix=pymupdf.Matrix(2, 2))   # convert image into high-resolution imag

    img = Image.open(io.BytesIO(pix.tobytes("png")))   # it take image as a PIl Image

    text = pytesseract.image_to_string(img)  # extract text from Tesseract OCR image 

    ocr_text += f"\n{text}"   # store extract in ocr_text

    print("=" * 100)
    print("Page:", page_num + 1)
    print("Characters:", len(text))
    print(text[:3000])

Page: 1
Characters: 2175
Pradhan Mantri Jan Dhan Yojana (PMJDY)

Highlights of the scheme :

1

Hon'ble Prime Minister will launch the National Mission on Financial Inclusion

named as Pradhan Mantri Jan Dhan Yojana (PMJDY) on 28" August, 2014.

2.
3.

The Mission will be implemented in two phases.
Phase-| from 15" August 2014 to 14" August 2015:

(i) Universal access to banking facilities for all households across the country
through a bank branch or a fixed point Business Correspondent (BC) within a
reasonable distance

(ii) To cover all households with atleast one Basic Banking Account with
RuPay Debit card having inbuilt accident insurance cover of Rs.1 lakh.
Further an overdraft facility upto Rs.5000 will also be permitted to Adhaar
enabled accounts after satisfactory operation in the account for 6 months.

(iii) Financial literacy programme which aims to take financial literacy upto
village level.

(iv) The Mission also envisages expansion of Direct Benefit Transfer under
various

PMJDY chunks: 3
Total chunks after PMJDY: 2037


In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# 1. Split documents into chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = splitter.split_documents(docs)

print("Total chunks:", len(chunks))

Total chunks: 2034


In [17]:
from langchain_core.documents import Document

pmjdy_doc = Document(
    page_content=ocr_text,
    metadata={
        "source": r"C:\Users\Admin\OneDrive\Documents\End_to_End_Project\Documents\PMJDY_Guidelines.pdf",
        "page": 0
    }
)

pmjdy_chunks = splitter.split_documents([pmjdy_doc])

print("PMJDY chunks:", len(pmjdy_chunks))

# Add PMJDY OCR chunks
chunks = chunks + pmjdy_chunks

print("Total chunks after PMJDY:", len(chunks))

PMJDY chunks: 3
Total chunks after PMJDY: 2040


In [10]:
# Split PMJDY OCR document
pmjdy_chunks = splitter.split_documents(
    [pmjdy_doc]
)

print("PMJDY chunks:", len(pmjdy_chunks))

PMJDY chunks: 3


In [11]:
base_chunks = chunks.copy()

# Remove PMJDY if already present
base_chunks = [
    doc for doc in base_chunks
    if os.path.basename(
        doc.metadata.get("source", "")
    ) != "PMJDY_Guidelines.pdf"
]

# Add PMJDY OCR chunks
final_chunks = base_chunks + pmjdy_chunks

print("Base chunks:", len(base_chunks))
print("PMJDY chunks:", len(pmjdy_chunks))
print("Final chunks:", len(final_chunks))

Base chunks: 2034
PMJDY chunks: 3
Final chunks: 2037


In [12]:
# 2. Multilingual embedding model
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

print("✅ Multilingual embedding model loaded!")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

✅ Multilingual embedding model loaded!


In [13]:
# 3. Create FAISS vector database
vectorstore = FAISS.from_documents(
    final_chunks,
    embeddings
)

print("✅ FAISS vector store created!")

✅ FAISS vector store created!


In [14]:
pmjdy_count = 0

for i in range(vectorstore.index.ntotal):

    doc = vectorstore.docstore.search(
        vectorstore.index_to_docstore_id[i]
    )

    if doc:

        source = os.path.basename(
            doc.metadata.get("source", "")
        )

        if source == "PMJDY_Guidelines.pdf":
            pmjdy_count += 1

print("PMJDY documents in FAISS:", pmjdy_count)

PMJDY documents in FAISS: 3


In [15]:
# 4. Create retriever
retriever = vectorstore.as_retriever(
    search_kwargs={"k": 6}
)

print("✅ Retriever ready!")

✅ Retriever ready!


In [22]:
# Test embedding with English question
test_embedding_en = embeddings.embed_query("Who is eligible for the PM-KISAN scheme?")

print("English embedding dimension:", len(test_embedding_en))

# Test embedding with Marathi question
test_embedding_mr = embeddings.embed_query("पीएम किसान योजनेसाठी कोण पात्र आहे?")

print("Marathi embedding dimension:", len(test_embedding_mr))


# Test embedding with Hindi question
test_embedding_hi = embeddings.embed_query("पीएम किसान योजना के लिए कौन पात्र है?")

print("Hindi embedding dimension:", len(test_embedding_hi))

English embedding dimension: 384
Marathi embedding dimension: 384
Hindi embedding dimension: 384


In [23]:
import os

def rerank_eligibility(results, top_k=4):

    candidates = []

    for doc, score in results:

        content = doc.page_content.lower()

        boost = 0

        # =====================================================
        # 1. STRONG SIGNAL - GENERAL PM-KISAN ELIGIBILITY
        # =====================================================

        if "income support to all landholding farmers" in content:
            boost -= 1.00

        if "landholding farmers’ families" in content:
            boost -= 0.60

        if "landholding farmers' families" in content:
            boost -= 0.60

        if "having cultivable land" in content:
            boost -= 0.30

        # =====================================================
        # 2. FARMER FAMILY DEFINITION
        # =====================================================

        if "husband, wife, and minor children" in content:
            boost -= 0.40

        # =====================================================
        # 3. PENALIZE SPECIAL / NON-GENERAL CONDITIONS
        # =====================================================

        if "due to inheritance" in content:
            boost += 0.50

        if "deceased land owner" in content:
            boost += 0.50

        if "reassessing the eligibility" in content:
            boost += 0.50

        # Manipur-specific condition
        if "manipur" in content:
            boost += 0.30

        # PM-KISAN portal / beneficiary database
        if "pm-kisan portal" in content:
            boost += 0.20

        # =====================================================
        # FINAL SCORE
        # Lower score = better result
        # =====================================================

        final_score = score + boost

        candidates.append(
            (doc, final_score, score)
        )

    # Sort by final reranked score
    candidates.sort(key=lambda x: x[1])

    # =========================================================
    # REMOVE DUPLICATE CONTENT
    # =========================================================

    unique = []
    seen = set()

    for doc, final_score, original_score in candidates:

        text_key = doc.page_content.strip()

        if text_key not in seen:

            seen.add(text_key)

            unique.append(
                (doc, original_score)
            )

        if len(unique) >= top_k:
            break

    return unique

In [27]:
def expand_query(question):

    q = question.lower().strip()

    # =========================================================
    # NAIS - National Agricultural Insurance Scheme
    # =========================================================
    if (
        "nais" in q
        or "राष्ट्रीय कृषि बीमा" in q
        or "राष्ट्रीय कृषी विमा" in q
        or "कृषि बीमा योजना" in q
        or "कृषी विमा योजना" in q
    ):
        return (
            question
            + " NAIS National Agricultural Insurance Scheme "
              "crop insurance agricultural insurance "
              "farmers crop loss crop failure yield loss "
              "indemnity insurance coverage financial compensation"
        )

    # =========================================================
    # Maharashtra Agriculture Schemes
    # =========================================================
    elif (
        "maharashtra agriculture" in q
        or "agriculture schemes" in q
        or "महाराष्ट्र कृषी" in q
        or "महाराष्ट्र कृषि" in q
        or "कृषी योजना" in q
        or "कृषि योजना" in q
    ):
        return (
            question
            + " Maharashtra agriculture schemes "
              "agriculture farmer schemes Maharashtra "
              "farmer assistance agricultural support "
              "crop agriculture financial assistance"
        )

    # =========================================================
    # PMAY - Pradhan Mantri Awas Yojana Urban
    # =========================================================
    elif (
        "pmay" in q
        or "pmay urban" in q
        or "पीएमएवाई" in q
        or "प्रधानमंत्री आवास" in q
        or "आवास योजना" in q
    ):
        return (
            question
            + " PMAY Urban Pradhan Mantri Awas Yojana Urban "
              "Housing for All urban housing beneficiary "
              "central assistance affordable housing "
              "housing support"
        )

    # =========================================================
    # PMFBY - Pradhan Mantri Fasal Bima Yojana
    # =========================================================
    elif (
        "pmfby" in q
        or "पीएमएफबीवाय" in q
        or "पीएमएफबीवाई" in q
        or "फसल बीमा" in q
        or "पीक विमा" in q
    ):
        return (
            question
            + " PMFBY Pradhan Mantri Fasal Bima Yojana "
              "crop insurance farmers crop loss crop damage "
              "financial support insurance coverage "
              "natural calamities pests diseases production risk"
        )

    # =========================================================
    # PMJDY - Pradhan Mantri Jan Dhan Yojana
    # =========================================================
    elif (
        "pmjdy" in q
        or "पीएमजेडीवाई" in q
        or "प्रधानमंत्री जन धन" in q
        or "जन धन" in q
        or "jan dhan" in q
    ):
        return (
            question
            + " PMJDY Pradhan Mantri Jan Dhan Yojana "
              "financial inclusion National Mission on Financial Inclusion "
              "universal access to banking facilities "
              "basic banking account RuPay debit card "
              "accident insurance overdraft financial literacy "
              "Direct Benefit Transfer DBT micro insurance pension "
              "banking facilities households"
            )

    # =========================================================
    # PMJJBY - Pradhan Mantri Jeevan Jyoti Bima Yojana
    # =========================================================
    elif (
        "pmjjby" in q
        or "पीएमजेजेबीवाय" in q
        or "प्रधानमंत्री जीवन ज्योति" in q
        or "जीवन ज्योति बीमा" in q
        or "jeevan jyoti" in q
    ):
        return (
            question
            + " PMJJBY Pradhan Mantri Jeevan Jyoti Bima Yojana "
              "life insurance scheme subscriber eligibility "
              "life cover death benefit premium insurance"
        )

    # =========================================================
    # PMKVY 4.0 - Pradhan Mantri Kaushal Vikas Yojana
    # =========================================================
    elif (
        "pmkvy" in q
        or "pmkvy 4.0" in q
        or "पीएमकेव्हीवाय" in q
        or "पीएमकेव्हीवाय 4.0" in q
        or "कौशल विकास" in q
        or "कौशल्य विकास" in q
        or "skill development" in q
    ):
        return (
            question
            + " PMKVY PMKVY 4.0 Pradhan Mantri Kaushal Vikas Yojana "
              "skill development training skill training "
              "candidates trainees certification job roles "
              "vocational training employment"
        )

    # =========================================================
    # PM-KISAN
    # =========================================================
    elif (
        "pm-kisan" in q
        or "pm kisan" in q
        or "pmकिसान" in q
        or "पीएम-किसान" in q
        or "पीएम किसान" in q
        or "pm-किसान" in q
        or "pm किसान" in q
    ):
        return (
            question
            + " PM-KISAN Pradhan Mantri Kisan Samman Nidhi "
              "financial support income support Rs 6000 per year "
              "landholding farmers farmer family cultivable land "
              "husband wife minor children eligibility"
        )

    # =========================================================
    # SDRF / NDRF - Disaster Response Funds
    # =========================================================
    elif (
        "sdrf" in q
        or "ndrf" in q
        or "एसडीआरएफ" in q
        or "एनडीआरएफ" in q
        or "disaster response" in q
        or "disaster relief" in q
        or "आपत्ती" in q
        or "आपदा" in q
    ):
        return (
            question
            + " SDRF State Disaster Response Fund "
              "NDRF National Disaster Response Fund "
              "disaster relief assistance natural calamity "
              "flood cyclone earthquake landslide drought "
              "disaster response"
        )

    # =========================================================
    # WBCIS - Weather Based Crop Insurance Scheme
    # =========================================================
    elif (
        "wbcis" in q
        or "weather based crop insurance" in q
        or "हवामान आधारित पीक विमा" in q
        or "weather based" in q
        or "हवामान" in q
    ):
        return (
            question
            + " WBCIS Weather Based Crop Insurance Scheme "
              "weather parameters rainfall temperature wind humidity "
              "weather triggers crop loss insurance coverage "
              "claim payout indemnity farmers"
        )

    # =========================================================
    # Default
    # =========================================================
    else:
        return question

In [28]:
def get_scheme_sources(question):

    q = question.lower().strip()

    # --------------------------------------------------
    # NAIS
    # --------------------------------------------------
    if (
        "nais" in q
        or "राष्ट्रीय कृषि बीमा" in q
        or "राष्ट्रीय कृषी विमा" in q
    ):
        return ["NAIS_Scheme.pdf"]


    # --------------------------------------------------
    # Maharashtra Agriculture Schemes
    # --------------------------------------------------
    if (
        "agriculture schemes" in q
        or "maharashtra agriculture" in q
        or "महाराष्ट्र कृषी" in q
        or "महाराष्ट्र कृषि" in q
        or "कृषी योजना" in q
        or "कृषि योजना" in q
        or "या योजनां" in q
        or "या योजना" in q
    ):
        return ["New_Agriculture_Schemes_Maharashtra.pdf"]


    # --------------------------------------------------
    # PMAY Urban
    # --------------------------------------------------
    if (
        "pmay" in q
        or "प्रधानमंत्री आवास" in q
        or "आवास योजना" in q
    ):
        return ["PMAY_Urban_Guidelines.pdf"]


    # --------------------------------------------------
    # PMFBY
    # --------------------------------------------------
    if (
        "pmfby" in q
        or "पीएमएफबीवाय" in q
        or "पीएमएफबीवाई" in q
        or "फसल बीमा" in q
        or "पीक विमा" in q
    ):
        return [
            "PMFBY_Revamped_OGs.pdf",
            "Revamped Operational Guidelines_17th August 2020.pdf"
        ]


    # --------------------------------------------------
    # PMJDY
    # --------------------------------------------------
    if (
        "pmjdy" in q
        or "जन धन" in q
        or "jan dhan" in q
        or "प्रधानमंत्री जन धन" in q
    ):
        return ["PMJDY_Guidelines.pdf"]


    # --------------------------------------------------
    # PMJJBY
    # --------------------------------------------------
    if (
        "pmjjby" in q
        or "जीवन ज्योति" in q
        or "jeevan jyoti" in q
        or "प्रधानमंत्री जीवन ज्योति" in q
    ):
        return ["PMJJBY_Rules.pdf"]


    # --------------------------------------------------
    # PMKVY
    # --------------------------------------------------
    if (
        "pmkvy" in q
        or "पीएमकेव्हीवाय" in q
        or "कौशल विकास" in q
        or "कौशल्य विकास" in q
        or "skill development" in q
    ):
        return ["PMKVY_4_0.pdf"]


    # --------------------------------------------------
    # PM-KISAN
    # --------------------------------------------------
    if (
        "pm-kisan" in q
        or "pm kisan" in q
        or "pm-किसान" in q
        or "pm किसान" in q
        or "पीएम-किसान" in q
        or "पीएम किसान" in q
    ):
        return [
            "PM_KISAN_Guidelines.pdf",
            "PM_KISAN_Operational_Guidelines.pdf"
        ]


    # # --------------------------------------------------
   # SDRF / NDRF
    if (
        "sdrf" in q
        or "ndrf" in q
        or "एसडीआरएफ" in q
        or "एनडीआरएफ" in q
        or "disaster response" in q
        or "disaster relief" in q
        or "आपत्ती" in q
        or "आपदा" in q
    ):
        return ["SDRFNDRFGuideline_20082024.pdf"]
    


    # --------------------------------------------------
    # WBCIS
    # --------------------------------------------------
    if (
        "wbcis" in q
        or "weather based crop insurance" in q
        or "हवामान आधारित पीक विमा" in q
        or "weather based" in q
        or "हवामान" in q
    ):

        # Operational question
        if "operational" in q:
            return ["WBCIS_Operational_Guidelines.pdf"]

        # General WBCIS question
        return ["WBCIS_Guidelines.pdf"]

 
    return None

In [31]:
def retrieve_documents(question, top_k=4):

    # Step 1: Expand query
    expanded_question = expand_query(question)

    # Step 2: Detect relevant source files
    allowed_sources = get_scheme_sources(question)

    # Step 3: Retrieve more candidates
    results = vectorstore.similarity_search_with_score(
        expanded_question,
        k=100
    )

    # Step 4: Filter by relevant source
    if allowed_sources:

        filtered_results = []

        for doc, score in results:

            source = os.path.basename(
                doc.metadata.get("source", "")
            )

            if source in allowed_sources:
                filtered_results.append((doc, score))

        results = filtered_results

    # Step 5: Remove duplicate chunks
    unique_results = []
    seen = set()

    for doc, score in results:

        text_key = doc.page_content.strip()

        if text_key not in seen:
            seen.add(text_key)
            unique_results.append((doc, score))

    # Step 6: Return top results
    final_results = unique_results[:top_k]

    return [doc for doc, score in final_results]

In [32]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_template("""
You are a multilingual Government Scheme RAG assistant.

Your job is to answer the user's question using ONLY the provided
context.

========================
IMPORTANT RULES
========================

1. Use ONLY the information present in the CONTEXT.

2. Do NOT use outside knowledge, memory, assumptions, or general
knowledge.

3. If the answer is clearly available in the context, ANSWER THE
QUESTION. Do NOT use the fallback response.

4. Use the most directly relevant passage from the context.

5. For a general eligibility question, use the general eligibility
information first.

6. Do NOT include state-specific conditions such as Manipur, Nagaland,
etc. unless the user explicitly asks about that state.

7. Do NOT include detailed exclusions, portal information, beneficiary
uploading procedures, or documentation unless the user specifically
asks for them.

8. Keep the answer focused on exactly what the user asked.

9. Do not add information that is not required to answer the question.

========================
LANGUAGE RULES
========================

Answer in the SAME LANGUAGE as the user's question.

English question → English answer.

Marathi question → Marathi answer.

Hindi question → Hindi answer.

Mixed Marathi/English → simple Marathi.

Mixed Hindi/English → simple Hindi.

========================
SOURCE / CITATION RULES
========================

Every factual answer must include the source and page number from
the CONTEXT.

IMPORTANT:

Use ONLY the source and page number explicitly shown in the CONTEXT.

NEVER invent a page number.

NEVER use a page number from memory.

NEVER cite a page that is not present in the CONTEXT.

If the context says:

[Source: PM_KISAN_Guidelines.pdf, Page: 2]

then the citation MUST use Page: 2.

English:
(Source: <document_name>, Page: <page_number>)

Marathi:
(स्रोत: <document_name>, पृष्ठ: <page_number>)

Hindi:
(स्रोत: <document_name>, पेज: <page_number>)

Do not cite any other page.

========================
FALLBACK RULE
========================

Use the following fallback ONLY when the CONTEXT genuinely does not
contain enough information to answer the question:

"I do not have enough information in the provided documents to answer
this question. Please consult the relevant government department or
authorized person for more clarity."

IMPORTANT:

If even one relevant passage in the CONTEXT directly answers the
question, DO NOT use the fallback.

NEVER give an answer followed by the fallback.

The response must be either:

A) A complete answer with citation

OR

B) The fallback response.

Never both.

========================
CONTEXT
========================

{context}

========================
QUESTION
========================

{question}

========================
FINAL CHECK
========================

Before answering, verify:

- Did I answer only from the context?
- Did I answer the actual question?
- Did I use the correct language?
- Did I use the exact source name?
- Did I use the exact page number shown in the context?
- Did I avoid irrelevant state-specific conditions?
- Did I avoid unnecessary exclusions?
- Did I avoid adding a fallback after a valid answer?

Now provide the final answer.
""")

In [33]:
# 3. Build the RAG chain (Replaces RetrievalQA)
# This chain: Fetches chunks -> Formats Prompt -> Sends to DeepSeek -> Cleans Output
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
import os

def format_docs(docs):
    formatted_docs = []

    for doc in docs:
        source = os.path.basename(
            doc.metadata.get("source", "Unknown")
        )

        page = doc.metadata.get("page", "Unknown")

        # PDF page index is usually 0-based
        if isinstance(page, int):
            page = page + 1

        formatted_docs.append(
            f"[Source: {source}, Page: {page}]\n"
            f"{doc.page_content}"
        )

    return "\n\n---\n\n".join(formatted_docs)


rag_chain = (
    {
        "context": lambda question: format_docs(
            retrieve_documents(question)
        ),
        "question": RunnablePassthrough()
    }
    | prompt
    | chat_model
    | StrOutputParser()
)

print("✅ RAG chain created successfully!")
#  retriever → fetches relevant chunks from your FAISS vector store.

# RunnablePassthrough() → passes the user’s raw question unchanged.

# |prompt Fills the template with the retrieved context and question.
# chat_model Sends the formatted prompt to our LLM (DeepSeek‑V3 via HuggingFace).

# | StrOutputParser()  Cleans the raw LLM output into a plain string.
#        Ensures you get a neat text answer (instead of JSON or structured output).

# '\' pipe operator chains components together

# HuggingFaceEndpoint --> It’s a LangChain connector class that  lets us use
#                        pretrained models from Hugging Face Hub inside your Python code.

#===============================================================================================
#  chat_model = ChatHuggingFace(llm=llm)  -->> implies  take the Hugging Face model I connected to (llm)
#                                   and wrap it in a chat‑friendly interface
#                                   so LangChain can treat it like a conversational model.

#  What it does
# llm is our Hugging Face model connection (DeepSeek‑V3 ).

# ChatHuggingFace takes that connection and makes it behave like a chat model:
# It understands prompts structured as conversations (roles like system, user, assistant).
# It works smoothly with ChatPromptTemplate and other LangChain chat utilities.
# It formats inputs/outputs in a way that fits multi‑turn dialogue.
#===================================

✅ RAG chain created successfully!


In [34]:
question = "पीएम-किसान योजना के लिए कौन पात्र है?"

answer = rag_chain.invoke(question)

print("Answer:")
print(answer)

Answer:
पीएम-किसान योजना के लिए पात्र व्यक्ति वे हैं जो देश में कृषि योग्य भूमि वाले सभी भूमिधारक किसान परिवारों के सदस्य हैं। यह योजना किसान परिवारों को विभिन्न आवश्यकताओं को पूरा करने और उचित फसल स्वास्थ्य सुनिश्चित करने के लिए वित्तीय सहायता प्रदान करती है।  

(स्रोत: PM_KISAN_Guidelines.pdf, पृष्ठ: 2)


In [18]:
question = "पीएम-किसान योजना के लिए कौन पात्र है?"

answer = rag_chain.invoke(question)

print("Answer:")
print(answer)

Answer:
पीएम-किसान योजना के लिए पात्र व्यक्ति वह है जो भूमि धारक किसान परिवार का सदस्य हो और जिसके पास खेती योग्य भूमि हो। यह परिवार पति, पत्नी और नाबालिग बच्चों से मिलकर बना होता है।  

(स्रोत: PM_KISAN_Guidelines.pdf, पृष्ठ: 2)


In [24]:
question = "PM-KISAN के तहत किसानों को कितनी राशि मिलती है?"

answer = rag_chain.invoke(question)

print("Answer:")
print(answer)

Answer:
PM-KISAN के तहत किसानों को प्रति वर्ष ₹6000 की राशि मिलती है।  
(स्रोत: PM_KISAN_Guidelines.pdf, पेज: 2)


In [19]:
question = "पीएम-किसान योजना के लिए कौन पात्र है?"

answer = rag_chain.invoke(question)

print("Answer:")
print(answer)

Answer:
पीएम-किसान योजना के लिए पात्र व्यक्ति वे हैं जो देश में कृषि योग्य भूमि वाले सभी भूमिधारक किसान परिवारों के सदस्य हैं। यह योजना केंद्र सरकार द्वारा कार्यान्वित की गई है।  

(स्रोत: PM_KISAN_Guidelines.pdf, पेज: 2)


In [30]:
question = "Who is eligible for the PM-KISAN scheme?"

answer = rag_chain.invoke(question)

print("Answer:")
print(answer)

Answer:
Landholding farmers’ families with cultivable land are eligible for the PM-KISAN scheme. The scheme provides income support to such families to supplement their financial needs for procuring inputs and ensuring proper crop health.  

(स्रोत: PM_KISAN_Guidelines.pdf, पेज: 2)


In [31]:
question = "What is the objective of PM-KISAN?"

answer = rag_chain.invoke(question)

print("Answer:")
print(answer)

Answer:
The objective of PM-KISAN is to supplement the financial needs of all landholding farmers’ families in procuring various inputs to ensure proper crop health and appropriate yields, commensurate with the anticipated farm income as well as for domestic needs.  
(Source: PM_KISAN_Guidelines.pdf, Page: 2)


In [44]:
question = "How much financial support is provided under the PM-KISAN scheme?"

answer = rag_chain.invoke(question)

print("Answer:")
print(answer)

Answer:
Under the PM-KISAN scheme, an amount of Rs. 6000 per year is provided as financial support to eligible farmers.  
(स्रोत: PM_KISAN_Guidelines.pdf, पेज: 2)


In [33]:
question = "पीएम-किसान योजनेसाठी कोण पात्र आहे?"

answer = rag_chain.invoke(question)

print("Answer:")
print(answer)

Answer:
पीएम-किसान योजनेसाठी सर्व जमीनधारक शेतकरी कुटुंबे पात्र आहेत, ज्यांच्याकडे कृषीय जमीन आहे. या योजनेअंतर्गत केंद्र सरकार पात्र शेतकऱ्यांच्या बँक खात्यात दरवर्षी ६००० रुपये थेट लाभ हस्तांतरण (DBT) मोडमध्ये जारी करते.  
(स्रोत: PM_KISAN_Guidelines.pdf, पृष्ठ: 2)


In [34]:
question = "प्रधानमंत्री किसान योजना के लिए कौन पात्र है?"

answer = rag_chain.invoke(question)

print("Answer:")
print(answer)

Answer:
प्रधानमंत्री किसान सम्मान निधि (PM-KISAN) योजना के लिए वे सभी किसान परिवार पात्र हैं जिनके पास खेती योग्य भूमि है। यह योजना देश के सभी भूमिधारक किसान परिवारों को आय सहायता प्रदान करने के लिए है।  

(स्रोत: PM_KISAN_Guidelines.pdf, पेज: 2)


In [29]:
question = "What is the objective of PMFBY?"

answer = rag_chain.invoke(question)

print("Answer:")
print(answer)

Answer:
The objective of PMFBY is to support sustainable production in the agriculture sector by:  
- Providing financial support to farmers suffering crop loss/damage arising out of unforeseen events.  
- Stabilizing the income of farmers to ensure their continuance in farming.  
- Encouraging farmers to adopt innovative and modern agricultural practices.  
- Ensuring credit worthiness of the farmers, crop diversification, and enhancing growth and competitiveness of the agriculture sector besides protecting farmers from production risks.  

(Source: Revamped Operational Guidelines_17th August 2020.pdf, Page: 10)


In [35]:
question = "PMFBY योजनेचा उद्देश काय आहे?"

answer = rag_chain.invoke(question)

print("Answer:")
print(answer)

Answer:
PMFBY योजनेचा उद्देश हा शेतकऱ्यांना पिकांच्या नुकसानीपासून संरक्षण देणे आणि त्यांना आर्थिक सुरक्षितता प्रदान करणे आहे. या योजनेअंतर्गत शेतकऱ्यांना पिकांच्या नुकसानीच्या बाबतीत प्रतिपूर्ती मिळते, ज्यामुळे त्यांना आर्थिकदृष्ट्या स्थिर राहता येते. (स्रोत: Revamped Operational Guidelines_17th August 2020.pdf, पृष्ठ: 34)


In [40]:
question = "What type of support does PMFBY provide to farmers?"

answer = rag_chain.invoke(question)

print("Answer:")
print(answer)

Answer:
PMFBY provides the following types of support to farmers:  
- Financial support to farmers suffering crop loss/damage arising out of unforeseen events.  
- Stabilizing the income of farmers to ensure their continuance in farming.  
- Encouraging farmers to adopt innovative and modern agricultural practices.  
- Ensuring credit worthiness of the farmers, crop diversification, and enhancing growth and competitiveness of the agriculture sector besides protecting farmers from production risks.  

(Source: PMFBY_Revamped_OGs.pdf, Page: 8)


In [46]:
question = "PMFBY शेतकऱ्यांना कोणत्या प्रकारची मदत देते?"

answer = rag_chain.invoke(question)

print("Question:")
print(question)

print("\nAnswer:")
print(answer)

Question:
PMFBY शेतकऱ्यांना कोणत्या प्रकारची मदत देते?

Answer:
PMFBY शेतकऱ्यांना विमा आवरणाच्या अंतर्गत आर्थिक मदत प्रदान करते. उदाहरणार्थ, जर एखाद्या पिकाची विमा रक्कम 30,000 रुपये असेल आणि विमा धोक्यामुळे प्रभावित क्षेत्रात 40% नुकसान झाले असेल, तर या आवरणाखाली देय दावा 12,000 रुपये असेल. हंगामाच्या शेवटी उत्पन्नातील कमतरता 60% असल्यास, दाव्याची रक्कम 18,000 रुपये असेल आणि हंगामाच्या शेवटी देय शिल्लक रक्कम 6,000 रुपये असेल.  
(स्रोत: Revamped Operational Guidelines_17th August 2020.pdf, पेज: 84)


In [53]:
question = "PM-किसान योजनेअंतर्गत दरवर्षी किती आर्थिक मदत मिळते?"

answer = rag_chain.invoke(question)

print("Question:")
print(question)

print("\nAnswer:")
print(answer)

Question:
PM-किसान योजनेअंतर्गत दरवर्षी किती आर्थिक मदत मिळते?

Answer:
PM-किसान योजनेअंतर्गत दरवर्षी ₹6000/- आर्थिक मदत मिळते.  
(स्रोत: PM_KISAN_Guidelines.pdf, पृष्ठ: 2)


In [54]:
question = "PMFBY शेतकऱ्यांना कोणत्या प्रकारची मदत देते?"

answer = rag_chain.invoke(question)

print("Question:")
print(question)

print("\nAnswer:")
print(answer)

Question:
PMFBY शेतकऱ्यांना कोणत्या प्रकारची मदत देते?

Answer:
PMFBY (प्रधानमंत्री फसल बीमा योजना) शेतकऱ्यांना फसल नुकसान झाल्यास आर्थिक मदत प्रदान करते. उदाहरणार्थ, जर एखाद्या फसलची विमा रक्कम ₹30,000 असेल आणि विमा केलेल्या जोखमीमुळे प्रभावित क्षेत्रात 40% नुकसान झाले असेल, तर या कव्हर अंतर्गत देय दावा ₹12,000 असेल. याशिवाय, हंगामाच्या शेवटी उत्पादनातील कमतरता असल्यास, अतिरिक्त रक्कम देखील दिली जाते. (स्रोत: Revamped Operational Guidelines_17th August 2020.pdf, पृष्ठ: 84)


In [55]:
question = "PMFBY किसानों को किस प्रकार की सहायता प्रदान करती है?"

answer = rag_chain.invoke(question)

print("Question:")
print(question)

print("\nAnswer:")
print(answer)

Question:
PMFBY किसानों को किस प्रकार की सहायता प्रदान करती है?

Answer:
PMFBY किसानों को प्राकृतिक आपदाओं, कीटों और बीमारियों के कारण अधिसूचित फसल की विफलता की स्थिति में बीमा कवरेज और वित्तीय सहायता प्रदान करती है। यह योजना किसानों की आय को स्थिर करने, उन्हें कृषि में जारी रखने के लिए प्रोत्साहित करने, नवीन और आधुनिक कृषि पद्धतियों को अपनाने के लिए प्रोत्साहित करने और कृषि क्षेत्र में ऋण प्रवाह सुनिश्चित करने का लक्ष्य रखती है।  
(स्रोत: New_Agriculture_Schemes_Maharashtra.pdf, पेज: 1)


In [39]:
question = "What is the objective of PMJDY?"

answer = rag_chain.invoke(question)

print("=" * 100)
print(answer)

The objective of PMJDY (Pradhan Mantri Jan Dhan Yojana) is to achieve universal access to banking facilities for all households across the country through a bank branch or a fixed point Business Correspondent (BC) within a reasonable distance, and to cover all households with at least one Basic Banking Account with a RuPay Debit card having an inbuilt accident insurance cover of Rs.1 lakh. Additionally, it aims to promote financial literacy up to the village level.

(स्रोत: PMJDY_Guidelines.pdf, पृष्ठ: 1)


In [37]:
question = "What is the objective of PMJDY?"

print(expand_query(question))
print("\nAllowed source:")
print(get_scheme_sources(question))

What is the objective of PMJDY? PMJDY Pradhan Mantri Jan Dhan Yojana financial inclusion National Mission on Financial Inclusion universal access to banking facilities basic banking account RuPay debit card accident insurance overdraft financial literacy Direct Benefit Transfer DBT micro insurance pension banking facilities households

Allowed source:
['PMJDY_Guidelines.pdf']


In [38]:
question = "NAIS योजनेअंतर्गत शेतकऱ्यांना कोणत्या प्रकारचे संरक्षण मिळते?"

answer = rag_chain.invoke(question)

print("Answer:")
print(answer)

Answer:
NAIS योजनेअंतर्गत शेतकऱ्यांना त्यांच्या पिकांच्या नुकसानीमुळे होणारे आर्थिक नुकसान कमी करण्यासाठी संरक्षण मिळते. हे संरक्षण विविध उत्पादन जोखमींमुळे पिकांच्या नुकसानी आणि नाशामुळे होणाऱ्या आर्थिक नुकसानाला कमी करण्यासाठी आहे.  

(स्रोत: NAIS_Scheme.pdf, पृष्ठ: 19)


In [36]:
question = "PMFBY योजना किसानों को किस प्रकार की सुरक्षा प्रदान करती है?"

answer = rag_chain.invoke(question)

print("Answer:")
print(answer)

Answer:
PMFBY योजना किसानों को फसल की पैदावार में होने वाले नुकसान के खिलाफ सुरक्षा प्रदान करती है। यह सुरक्षा सूखा, बाढ़, कीट और बीमारी के हमले, भूस्खलन, प्राकृतिक आग, तूफान, ओलावृष्टि और चक्रवात जैसे गैर-रोकथाम योग्य जोखिमों के कारण होने वाले नुकसान को कवर करती है। यह सुरक्षा फसल की बुवाई से लेकर कटाई तक के समय के लिए प्रदान की जाती है। (स्रोत: Revamped Operational Guidelines_17th August 2020.pdf, पेज: 14)


# Hallucination Test

In [35]:
question = "What is the current interest rate offered on PMJDY bank accounts?"

answer = rag_chain.invoke(question)

print("Answer:")
print(answer)

Answer:
I do not have enough information in the provided documents to answer this question. Please consult the relevant government department or authorized person for more clarity.


In [41]:
question = "What is the customer care mobile number for PMFBY?"

answer = rag_chain.invoke(question)

print("Answer:")
print(answer)

Answer:
I do not have enough information in the provided documents to answer this question. Please consult the relevant government department or authorized person for more clarity.


In [42]:
# Save 

In [43]:
print("=" * 80)
print("FINAL PROJECT VERIFICATION")
print("=" * 80)

print("1. Total PDF pages loaded :", len(docs))
print("2. Normal PDF chunks      :", len(chunks))
print("3. PMJDY OCR chunks       :", len(pmjdy_chunks))
print("4. Final chunks           :", len(final_chunks))
print("5. FAISS index size      :", vectorstore.index.ntotal)
print("6. PMJDY in FAISS        :", pmjdy_count)
print("7. Retriever              :", "Ready" if retriever else "Not Ready")
print("8. RAG Chain              :", "Ready" if rag_chain else "Not Ready")

print("=" * 80)

FINAL PROJECT VERIFICATION
1. Total PDF pages loaded : 648
2. Normal PDF chunks      : 2040
3. PMJDY OCR chunks       : 3
4. Final chunks           : 2037
5. FAISS index size      : 2037
6. PMJDY in FAISS        : 3
7. Retriever              : Ready
8. RAG Chain              : Ready


In [44]:
# Save FAISS vector database

faiss_folder = "faiss_index"

vectorstore.save_local(faiss_folder)

print("✅ FAISS index saved successfully!")
print("Saved folder:", faiss_folder)

✅ FAISS index saved successfully!
Saved folder: faiss_index


In [45]:
# verify faiss save or not
import os

print("FAISS folder exists:", os.path.exists("faiss_index"))
print("Files:", os.listdir("faiss_index"))

FAISS folder exists: True
Files: ['index.faiss', 'index.pkl']
